# Revised Training (Classification + Segmentation)

Same step-by-step style as `hands_on.ipynb`, with the known problems fixed:

| Problem in the old notebooks | Fix here |
|---|---|
| Segmentation validation images were partly training images (each call made a new random split) | **One** fixed train/val split, saved to `split.json` and used by every step |
| `classification/model.py` was corrupted (SyntaxError) | `model.py` is written **first**, and training imports the network from it, so the file that gets submitted is the one that was trained |
| Inference preprocessing differed from training (Resize 336 + crop 320, 448 vs 384) | `model.py` holds one `eval_transform`; validation and `predict()` both use it |
| Checkpoint chosen with tuned thresholds / macro F1 | Chosen by **mean per-image F1 at threshold 0.5** (the Kaggle metric) |
| mIoU computed differently in each notebook | One shared metric: dataset-level IoU per class (classes 1–20), averaged over classes present |
| Several cells overwrote each other's weight files | Each model has its own file; `hands_on` files are not touched |
| Silent fallback to random weights if ImageNet download fails | Raises an error instead |
| Upside-down (vertical flip) augmentation | Removed |

**Run order:** top to bottom, from the repo root. Set `SMOKE_TEST = True` first to check that everything runs in a few minutes on a tiny subset.

## 0. Setup

In [ ]:
# Setup: imports, paths, settings
import importlib.util
import json
import os
import random
import shutil
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# -------------------------------------------------------------
# 1. Quick check mode
# -------------------------------------------------------------
# True  -> tiny subset, 1 epoch, small images: checks the whole notebook runs (a few minutes on CPU)
# False -> the real training run (use a GPU, e.g. Kaggle)
SMOKE_TEST = False

# -------------------------------------------------------------
# 2. Device & Seed Configuration
# -------------------------------------------------------------
SEED = 42

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'   # mixed precision only on GPU
NUM_WORKERS = 0 if os.name == "nt" else 2   # Windows notebooks cannot use worker processes

def make_scaler():
    # Gradient scaler for mixed precision (does nothing on CPU); works on old and new torch versions
    if hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    return torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f"Using device: {device}")

if device.type == 'cpu':
    torch.set_num_threads(min(16, os.cpu_count() or 4))

# -------------------------------------------------------------
# 3. Paths
# -------------------------------------------------------------
ROOT = Path(".").resolve()
KAGGLE = Path("/kaggle/input").exists()

if KAGGLE:
    found = [p for p in Path("/kaggle/input").rglob("Dataset")
             if (p / "train" / "labels.csv").exists() and (p / "test" / "images").exists()]
    if not found:
        raise FileNotFoundError("Attach the A2 dataset: no Dataset/ folder with train/ and test/ under /kaggle/input")
    DATA_DIR = found[0]
    OUT_DIR = Path("/kaggle/working") / "Roll_no_name_Assignment_2"
else:
    from data_utils import ensure_dataset
    DATA_DIR = ensure_dataset(ROOT / "data")   # downloads from Google Drive if missing
    OUT_DIR = ROOT / "outputs" / "Roll_no_name_Assignment_2"

if SMOKE_TEST:
    OUT_DIR = OUT_DIR.parent / "smoke_test"

TRAIN_IMG_DIR = DATA_DIR / "train" / "images"
TRAIN_MASK_DIR = DATA_DIR / "train" / "segmentation_masks"
LABELS_CSV = DATA_DIR / "train" / "labels.csv"
TEST_IMG_DIR = DATA_DIR / "test" / "images"

# Submission folder layout from the README
CLS_DIR = OUT_DIR / "classification"
SEG_DIR = OUT_DIR / "segmentation"
CLS_WEIGHTS = CLS_DIR / "weights"
SEG_WEIGHTS = SEG_DIR / "weights"
for d in (CLS_WEIGHTS, SEG_WEIGHTS):
    d.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------
# 4. Classes & image sizes
# -------------------------------------------------------------
CLASSES: List[str] = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
NUM_SEG_CLASSES = 21   # 0 = background + 20 objects (255 = ignore)

CLS_IMAGE_SIZE = 128 if SMOKE_TEST else 320
SEG_IMAGE_SIZE = 128 if SMOKE_TEST else 384

print("Data:   ", DATA_DIR)
print("Output: ", OUT_DIR)

In [ ]:
# ImageNet ResNet-50 backbone (allowed by the assignment).
# If the download fails (e.g. Kaggle without internet) it looks for the torchvision file
# 'resnet50-11ad3fa6.pth' in ./pretrained or an attached Kaggle dataset.
# It never falls back to random weights silently.
RESNET50_FILE = "resnet50-11ad3fa6.pth"

def get_pretrained_resnet50() -> nn.Module:
    try:
        return models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    except Exception as e:
        for folder in (ROOT / "pretrained", Path("/kaggle/input")):
            if folder.exists():
                for f in folder.rglob(RESNET50_FILE):
                    net = models.resnet50(weights=None)
                    net.load_state_dict(torch.load(f, map_location="cpu"))
                    print("Loaded ImageNet weights from", f)
                    return net
        raise RuntimeError(
            f"Could not download ImageNet ResNet-50 weights and no local {RESNET50_FILE} was found. "
            "Training from random weights would give a much worse model, so stopping here."
        ) from e


def load_module(name: str, path: Path):
    # Import a model.py file by path (always re-reads the file from disk)
    spec = importlib.util.spec_from_file_location(name, str(path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. One fixed train / validation split

Made once and saved to `split.json`. Classification, both segmentation models, the A-vs-B test and the bootstrap all use these **same** validation images, and no model ever trains on them.

In [ ]:
VAL_FRACTION = 0.15
SPLIT_FILE = OUT_DIR / "split.json"

all_ids = sorted(pd.read_csv(LABELS_CSV)["image_id"])
missing_masks = [i for i in all_ids if not (TRAIN_MASK_DIR / f"{i}.png").exists()]
assert not missing_masks, f"Images without a mask: {missing_masks[:5]}"

if SPLIT_FILE.exists():
    split = json.loads(SPLIT_FILE.read_text())
    print("Loaded existing split from", SPLIT_FILE)
else:
    ids = all_ids.copy()
    random.Random(SEED).shuffle(ids)
    n_val = int(VAL_FRACTION * len(ids))
    split = {"seed": SEED, "val": sorted(ids[:n_val]), "train": sorted(ids[n_val:])}
    SPLIT_FILE.write_text(json.dumps(split, indent=1))
    print("Created new split ->", SPLIT_FILE)

TRAIN_IDS: List[str] = split["train"]
VAL_IDS: List[str] = split["val"]
assert not set(TRAIN_IDS) & set(VAL_IDS), "train and val overlap!"

if SMOKE_TEST:
    TRAIN_IDS, VAL_IDS = TRAIN_IDS[:40], VAL_IDS[:16]

print(f"Split: {len(TRAIN_IDS)} Train | {len(VAL_IDS)} Validation")

## 2. Shared metrics (used everywhere below)

- **Classification:** F1 for each image (predicted class set vs. true class set), then the mean over images. Threshold 0.5. This is the Kaggle metric.
- **Segmentation:** add up intersection and union for each class over all images, compute IoU for each class 1–20, and average over the classes that appear. Per-image mIoU (needed for the Wilcoxon test) uses the same formula on one image.
- **RLE:** the README's format (`start length value` triplets), vectorised with NumPy so it's fast.

In [ ]:
# -------------------------------------------------------------
# 1. Classification: mean per-image F1
# -------------------------------------------------------------
def f1_per_image(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    # y_true, y_pred: (N, 20) arrays of 0/1. Returns one F1 per image.
    tp = ((y_true == 1) & (y_pred == 1)).sum(axis=1)
    fp = ((y_true == 0) & (y_pred == 1)).sum(axis=1)
    fn = ((y_true == 1) & (y_pred == 0)).sum(axis=1)
    denom = 2 * tp + fp + fn
    return np.where(denom == 0, 1.0, 2 * tp / np.maximum(denom, 1))

# -------------------------------------------------------------
# 2. Segmentation: IoU counts -> mIoU
# -------------------------------------------------------------
def iou_counts(pred: np.ndarray, gt: np.ndarray, num_classes: int = NUM_SEG_CLASSES):
    # Per-class intersection and union for one image (or a whole batch); 255 pixels are ignored
    valid = gt != 255
    p = pred[valid].astype(np.int64)
    g = gt[valid].astype(np.int64)
    inter = np.bincount(g[p == g], minlength=num_classes)[:num_classes]
    area_p = np.bincount(p, minlength=num_classes)[:num_classes]
    area_g = np.bincount(g, minlength=num_classes)[:num_classes]
    return inter, area_p + area_g - inter

def miou_from_counts(inter: np.ndarray, union: np.ndarray) -> float:
    # Mean IoU over object classes 1-20 that appear (in prediction or ground truth)
    inter, union = inter[1:], union[1:]
    present = union > 0
    return float(np.mean(inter[present] / union[present])) if present.any() else float("nan")

# -------------------------------------------------------------
# 3. Submission RLE (format from the README)
# -------------------------------------------------------------
def mask_to_rle(mask: np.ndarray) -> str:
    flat = np.asarray(mask, dtype=np.int64).reshape(-1)        # row-major
    change = np.flatnonzero(np.diff(flat)) + 1                  # where the value changes
    starts = np.concatenate([[0], change])
    lengths = np.concatenate([change, [flat.size]]) - starts
    values = flat[starts]
    keep = values != 0                                          # background is implicit
    parts = np.stack([starts[keep], lengths[keep], values[keep]], axis=1).reshape(-1)
    return " ".join(map(str, parts)) if parts.size else "0 0 0"

def rle_to_mask(rle_string: str, height: int, width: int) -> np.ndarray:
    mask = np.zeros(height * width, dtype=np.uint8)
    parts = list(map(int, rle_string.split()))
    for i in range(0, len(parts), 3):
        start, length, value = parts[i], parts[i + 1], parts[i + 2]
        mask[start:start + length] = value
    return mask.reshape(height, width)

# -------------------------------------------------------------
# 4. Quick self-checks
# -------------------------------------------------------------
assert mask_to_rle(np.array([[0, 3, 3], [3, 0, 5]])) == "1 3 3 5 1 5"
_m = np.random.default_rng(0).integers(0, 4, size=(37, 53)).astype(np.uint8)
assert np.array_equal(rle_to_mask(mask_to_rle(_m), 37, 53), _m)
assert np.allclose(f1_per_image(np.array([[1, 0, 1]]), np.array([[1, 1, 0]])), [0.5])
_i, _u = iou_counts(np.array([1, 1, 2, 0]), np.array([1, 2, 2, 255]))
assert miou_from_counts(_i, _u) == 0.5   # class 1: 1/2, class 2: 1/2
print("Metric self-checks passed.")

# Classification

First write `classification/model.py`, then train **using the network and preprocessing from that file**. The submitted file is therefore exactly the code that was trained and validated.

In [ ]:
# Write classification/model.py (the submitted file)
CLASSIFICATION_MODEL_PY = '''
# classification/model.py - ResNet-50 (ImageNet backbone) + custom 2-layer MLP head.
# Written by revised_training.ipynb. The same eval_transform is used for validation and predict().
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
IMAGE_SIZE = __IMAGE_SIZE__
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])


class ClassifierNet(nn.Module):
    def __init__(self, num_classes: int = 20, backbone: nn.Module = None):
        super().__init__()
        if backbone is None:                      # inference: trained weights are loaded afterwards
            backbone = models.resnet50(weights=None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])   # drop avgpool + fc
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(                # our own head
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.pool(self.backbone(x)).flatten(1))


class ClassificationModel:
    def __init__(self, weights_dir: str):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = ClassifierNet(num_classes=len(CLASSES))
        ckpt = torch.load(Path(weights_dir) / 'best_classifier.pth', map_location='cpu')
        self.model.load_state_dict(ckpt['model_state'])
        self.model.to(self.device).eval()

    def predict(self, image: np.ndarray) -> dict:
        img = Image.fromarray(np.asarray(image, dtype=np.uint8)).convert('RGB')
        x = eval_transform(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            probs = torch.sigmoid(self.model(x))[0].cpu().numpy()
        return {name: float(p) for name, p in zip(CLASSES, probs)}
'''

(CLS_DIR / "model.py").write_text(
    CLASSIFICATION_MODEL_PY.replace("__IMAGE_SIZE__", str(CLS_IMAGE_SIZE)), encoding="utf-8")
cls_module = load_module("cls_model", CLS_DIR / "model.py")
print("Wrote", CLS_DIR / "model.py")

In [ ]:
# Actual training - Classification Model (revised)

# -------------------------------------------------------------
# 1. Settings
# -------------------------------------------------------------
EPOCHS_CLS = 1 if SMOKE_TEST else 25
BATCH_SIZE_CLS = 8 if SMOKE_TEST else (32 if device.type == 'cuda' else 16)
LR_CLS = 3e-4

# -------------------------------------------------------------
# 2. Dataset (same as hands_on, but built from the fixed split)
# -------------------------------------------------------------
class MultiLabelDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.df[CLASSES].values.astype(np.float32)
        self.image_ids = self.df['image_id'].values

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        with Image.open(self.img_dir / f"{self.image_ids[idx]}.jpg") as img:
            image = img.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.from_numpy(self.labels[idx])

# -------------------------------------------------------------
# 3. Transforms: validation uses model.py's eval_transform
# -------------------------------------------------------------
train_transforms = transforms.Compose([
    transforms.Resize((CLS_IMAGE_SIZE + 32, CLS_IMAGE_SIZE + 32)),
    transforms.RandomResizedCrop(CLS_IMAGE_SIZE, scale=(0.6, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(p=0.5),          # no vertical flip: objects are rarely upside down
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=cls_module.MEAN, std=cls_module.STD),
])
val_transforms = cls_module.eval_transform

df_all = pd.read_csv(LABELS_CSV)
train_df = df_all[df_all['image_id'].isin(TRAIN_IDS)]
val_df = df_all[df_all['image_id'].isin(VAL_IDS)]

train_dataset = MultiLabelDataset(train_df, TRAIN_IMG_DIR, transform=train_transforms)
val_dataset = MultiLabelDataset(val_df, TRAIN_IMG_DIR, transform=val_transforms)
# drop_last: BatchNorm1d in the head cannot train on a batch of 1
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_CLS, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_CLS, shuffle=False, num_workers=NUM_WORKERS)
print(f"Classification: {len(train_dataset)} Train | {len(val_dataset)} Validation")

# -------------------------------------------------------------
# 4. Model: network class comes from model.py
# -------------------------------------------------------------
set_seed()
model = cls_module.ClassifierNet(num_classes=len(CLASSES), backbone=get_pretrained_resnet50()).to(device)

# -------------------------------------------------------------
# 5. Loss, Optimizer & Scheduler
# -------------------------------------------------------------
# Mild pos_weight for rare classes (sqrt of neg/pos, capped at 5)
pos = train_df[CLASSES].values.sum(axis=0)
neg = len(train_df) - pos
pos_weight = torch.tensor(np.sqrt((neg + 1) / (pos + 1)), dtype=torch.float32).clamp(1.0, 5.0).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_CLS, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_CLS, eta_min=1e-6)
scaler = make_scaler()

# -------------------------------------------------------------
# 6. Training Loop: keep the epoch with the best per-image F1 at 0.5
# -------------------------------------------------------------
best_val_f1 = -1.0
best_cls_path = CLS_WEIGHTS / "best_classifier.pth"
cls_history = []

print(f"\nStarting training for {EPOCHS_CLS} epochs...\n" + "-" * 60)
for epoch in range(1, EPOCHS_CLS + 1):
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            loss = criterion(model(images), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * images.size(0)
    train_loss /= max(1, len(train_loader) * BATCH_SIZE_CLS)
    scheduler.step()

    # Validation
    model.eval()
    all_targets, all_probs = [], []
    with torch.no_grad():
        for images, targets in val_loader:
            all_probs.append(torch.sigmoid(model(images.to(device))).float().cpu().numpy())
            all_targets.append(targets.numpy())
    y_true = np.vstack(all_targets).astype(int)
    y_pred = (np.vstack(all_probs) >= 0.5).astype(int)
    val_f1 = float(f1_per_image(y_true, y_pred).mean())
    cls_history.append({"epoch": epoch, "train_loss": train_loss, "val_f1": val_f1})

    is_best = val_f1 > best_val_f1
    if is_best:
        best_val_f1 = val_f1
        torch.save({'model_state': model.state_dict(), 'val_f1': val_f1, 'epoch': epoch}, best_cls_path)

    best_mark = " [BEST SAVED]" if is_best else ""
    print(f"Epoch [{epoch:02d}/{EPOCHS_CLS:02d}] | Train Loss: {train_loss:.4f} | Val per-image F1@0.5: {val_f1:.4f}{best_mark}")

print("-" * 60)
print(f"Training Complete! Best Validation F1: {best_val_f1:.4f}")
print(f"Saved weights to: {best_cls_path}")

In [ ]:
# model testing: classification (through the submitted model.py, exactly as the grader runs it)
clf = load_module("cls_model_check", CLS_DIR / "model.py").ClassificationModel(weights_dir=str(CLS_WEIGHTS))

# 1. One sample image, like hands_on
sample_id = VAL_IDS[0]
with Image.open(TRAIN_IMG_DIR / f"{sample_id}.jpg") as img:
    img_arr = np.array(img.convert("RGB"))
probs_dict = clf.predict(img_arr)
assert len(probs_dict) == 20
predicted_classes = [c for c, p in probs_dict.items() if p >= 0.5]
true_classes = [c for c in CLASSES if val_df.set_index('image_id').loc[sample_id, c] == 1]
print(f"Image: {sample_id} | true: {true_classes}")
for c, p in sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  - {c:12s}: {p:.4f}")
print(f"Final Classification Output: '{' '.join(predicted_classes) if predicted_classes else 'background'}'")

# 2. Whole validation set through predict(): should match the best F1 from training
preds = []
for image_id in val_df['image_id']:
    with Image.open(TRAIN_IMG_DIR / f"{image_id}.jpg") as img:
        p = clf.predict(np.array(img.convert("RGB")))
    preds.append([int(p[c] >= 0.5) for c in CLASSES])
f1_via_model_py = float(f1_per_image(val_df[CLASSES].values.astype(int), np.array(preds)).mean())
print(f"\nVal per-image F1 via model.py: {f1_via_model_py:.4f}  (training reported {best_val_f1:.4f})")

# Segmentation

#### The "255" ignore label
In PASCAL VOC masks, object boundaries are marked 255 (void). The network has 21 output channels (0 = background, 1–20 = objects), so 255 is not a class. `nn.CrossEntropyLoss(ignore_index=255)` and the metrics skip those pixels.

Two models are trained **on the same training images** and compared **on the same validation images**. They differ only in the loss:
- **Model A:** cross-entropy + Dice
- **Model B:** cross-entropy only

In [ ]:
# Write segmentation/model.py (the submitted file)
SEGMENTATION_MODEL_PY = '''
# segmentation/model.py - ResNet-50 (ImageNet backbone) + custom multi-scale skip decoder.
# Written by revised_training.ipynb. The same eval_transform is used for validation and predict().
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models, transforms

NUM_CLASSES = 21          # 0 = background, 1-20 = objects
IMAGE_SIZE = __IMAGE_SIZE__
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])


class ConvBNAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class CustomSegmentationNet(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, backbone: nn.Module = None):
        super().__init__()
        if backbone is None:                      # inference: trained weights are loaded afterwards
            backbone = models.resnet50(weights=None)
        # Encoder (ResNet-50)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1  # 256 channels  (1/4)
        self.layer2 = backbone.layer2  # 512 channels  (1/8)
        self.layer3 = backbone.layer3  # 1024 channels (1/16)
        self.layer4 = backbone.layer4  # 2048 channels (1/32)
        # Our decoder: fuse deep features with skip connections, step by step
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1)
        self.fuse34 = nn.Sequential(ConvBNAct(256 + 1024, 256), ConvBNAct(256, 256))
        self.fuse23 = nn.Sequential(ConvBNAct(256 + 512, 192), ConvBNAct(192, 192))
        self.fuse12 = nn.Sequential(ConvBNAct(192 + 256, 128), ConvBNAct(128, 128))
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = x.shape[2], x.shape[3]
        c1 = self.layer1(self.stem(x))
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)

        d = self.proj4(c4)
        d = F.interpolate(d, size=c3.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse34(torch.cat([d, c3], dim=1))
        d = F.interpolate(d, size=c2.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse23(torch.cat([d, c2], dim=1))
        d = F.interpolate(d, size=c1.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse12(torch.cat([d, c1], dim=1))

        logits = self.classifier(d)
        return F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)


class SegmentationModel:
    def __init__(self, weights_dir: str, checkpoint: str = 'best_segmentation.pth'):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = CustomSegmentationNet(num_classes=NUM_CLASSES)
        ckpt = torch.load(Path(weights_dir) / checkpoint, map_location='cpu')
        self.model.load_state_dict(ckpt['model_state'])
        self.model.to(self.device).eval()

    def predict(self, image: np.ndarray) -> np.ndarray:
        h, w = image.shape[:2]
        img = Image.fromarray(np.asarray(image, dtype=np.uint8)).convert('RGB')
        x = eval_transform(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.model(x) + torch.flip(self.model(torch.flip(x, dims=[3])), dims=[3])  # flip TTA
            logits = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)  # back to original size
            pred = torch.argmax(logits, dim=1)[0]
        return pred.cpu().numpy().astype(np.uint8)
'''

(SEG_DIR / "model.py").write_text(
    SEGMENTATION_MODEL_PY.replace("__IMAGE_SIZE__", str(SEG_IMAGE_SIZE)), encoding="utf-8")
seg_module = load_module("seg_model", SEG_DIR / "model.py")
print("Wrote", SEG_DIR / "model.py")

In [ ]:
# Actual training - segmentation model (revised)

# -------------------------------------------------------------
# 1. Settings
# -------------------------------------------------------------
EPOCHS_SEG = 1 if SMOKE_TEST else 30
BATCH_SIZE_SEG = 4 if SMOKE_TEST else (8 if device.type == 'cuda' else 4)   # lower to 4 if the GPU runs out of memory

# -------------------------------------------------------------
# 2. Segmentation Dataset with Nearest-Neighbor Masks
# -------------------------------------------------------------
class SegmentationDataset(Dataset):
    def __init__(self, image_ids: List[str], img_size: int = SEG_IMAGE_SIZE, is_train: bool = True):
        self.image_ids = image_ids
        self.img_size = img_size
        self.is_train = is_train
        self.color_jitter = transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2)
        self.to_tensor_norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=seg_module.MEAN, std=seg_module.STD),
        ])

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        image_id = self.image_ids[idx]
        img = Image.open(TRAIN_IMG_DIR / f"{image_id}.jpg").convert("RGB")
        mask = Image.open(TRAIN_MASK_DIR / f"{image_id}.png")

        if self.is_train:
            # Resize a bit larger, then random crop (Bilinear for image, NEAREST for the class mask!)
            big = self.img_size + 64
            img = img.resize((big, big), Image.Resampling.BILINEAR)
            mask = mask.resize((big, big), Image.Resampling.NEAREST)
            left, top = random.randint(0, 64), random.randint(0, 64)
            box = (left, top, left + self.img_size, top + self.img_size)
            img, mask = img.crop(box), mask.crop(box)
            if random.random() < 0.5:     # horizontal flip only
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            img = self.color_jitter(img)
            img_tensor = self.to_tensor_norm(img)
        else:
            img_tensor = seg_module.eval_transform(img)      # same as predict()
            mask = mask.resize((self.img_size, self.img_size), Image.Resampling.NEAREST)

        mask_tensor = torch.from_numpy(np.array(mask, dtype=np.int64))
        return img_tensor, mask_tensor

train_seg_dataset = SegmentationDataset(TRAIN_IDS, is_train=True)
val_seg_dataset = SegmentationDataset(VAL_IDS, is_train=False)
print(f"Segmentation: {len(train_seg_dataset)} Train | {len(val_seg_dataset)} Validation")

# -------------------------------------------------------------
# 3. Losses: Cross-Entropy (+ optional Soft Dice)
# -------------------------------------------------------------
class DiceLoss(nn.Module):
    def __init__(self, num_classes: int = 21, ignore_index: int = 255, smooth: float = 1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits.float(), dim=1)
        valid_mask = (targets != self.ignore_index)
        targets_clean = torch.where(valid_mask, targets, torch.zeros_like(targets))
        targets_one_hot = F.one_hot(targets_clean, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        valid_mask_exp = valid_mask.unsqueeze(1).float()
        probs = probs * valid_mask_exp
        targets_one_hot = targets_one_hot * valid_mask_exp

        intersection = torch.sum(probs * targets_one_hot, dim=(0, 2, 3))
        cardinality = torch.sum(probs + targets_one_hot, dim=(0, 2, 3))
        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - torch.mean(dice_score)

criterion_ce = nn.CrossEntropyLoss(ignore_index=255)
criterion_dice = DiceLoss(num_classes=NUM_SEG_CLASSES, ignore_index=255)

# -------------------------------------------------------------
# 4. Training & Validation Loop (one function, used for model A and B)
# -------------------------------------------------------------
def train_segmentation(name: str, dice_weight: float, lr: float = 2e-4, epochs: int = EPOCHS_SEG) -> Path:
    set_seed()   # same seed for A and B: only the loss differs
    train_loader = DataLoader(train_seg_dataset, batch_size=BATCH_SIZE_SEG, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    val_loader = DataLoader(val_seg_dataset, batch_size=BATCH_SIZE_SEG, shuffle=False, num_workers=NUM_WORKERS)

    model = seg_module.CustomSegmentationNet(num_classes=NUM_SEG_CLASSES, backbone=get_pretrained_resnet50()).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = make_scaler()

    best_val_miou = -1.0
    best_path = SEG_WEIGHTS / f"segmentation_{name}.pth"
    print(f"\n[Model {name}] dice_weight={dice_weight}, lr={lr}, {epochs} epochs\n" + "-" * 70)

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        train_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                logits = model(images)
                loss = criterion_ce(logits, masks)
                if dice_weight > 0:
                    loss = loss + dice_weight * criterion_dice(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        train_loss /= max(1, len(train_loader))
        scheduler.step()

        # Validation: dataset-level mIoU (classes 1-20)
        model.eval()
        inter_sum = np.zeros(NUM_SEG_CLASSES)
        union_sum = np.zeros(NUM_SEG_CLASSES)
        with torch.no_grad():
            for images, masks in val_loader:
                preds = torch.argmax(model(images.to(device)), dim=1).cpu().numpy()
                inter, union = iou_counts(preds, masks.numpy())
                inter_sum += inter
                union_sum += union
        val_miou = miou_from_counts(inter_sum, union_sum)

        is_best = val_miou > best_val_miou
        if is_best:
            best_val_miou = val_miou
            torch.save({'model_state': model.state_dict(), 'val_miou': val_miou, 'epoch': epoch,
                        'dice_weight': dice_weight, 'lr': lr}, best_path)
        best_mark = " [BEST SAVED]" if is_best else ""
        print(f"Epoch [{epoch:02d}/{epochs:02d}] | Train Loss: {train_loss:.4f} | Val mIoU: {val_miou*100:.2f}%{best_mark}")

    print(f"[Model {name}] Best Validation mIoU: {best_val_miou*100:.2f}%  -> {best_path}")
    return best_path

In [ ]:
# Train the two segmentation models (same data, same seed; only the loss differs)
seg_path_a = train_segmentation("A", dice_weight=0.5)   # Cross-Entropy + Dice
seg_path_b = train_segmentation("B", dice_weight=0.0)   # Cross-Entropy only

In [ ]:
# Evaluate A and B through the submitted model.py, at ORIGINAL image size (as Kaggle scores it)
def evaluate_segmentation(checkpoint_name: str):
    seg = seg_module.SegmentationModel(str(SEG_WEIGHTS), checkpoint=checkpoint_name)
    inters, unions = [], []
    for image_id in VAL_IDS:
        with Image.open(TRAIN_IMG_DIR / f"{image_id}.jpg") as img:
            image = np.array(img.convert("RGB"))
        gt = np.array(Image.open(TRAIN_MASK_DIR / f"{image_id}.png"))
        pred = seg.predict(image)
        assert pred.shape == gt.shape
        i, u = iou_counts(pred, gt)
        inters.append(i)
        unions.append(u)
    return np.array(inters), np.array(unions)   # (num_val_images, 21) each

inter_a, union_a = evaluate_segmentation(seg_path_a.name)
inter_b, union_b = evaluate_segmentation(seg_path_b.name)

miou_a = miou_from_counts(inter_a.sum(0), union_a.sum(0))
miou_b = miou_from_counts(inter_b.sum(0), union_b.sum(0))
per_image_a = np.array([miou_from_counts(i, u) for i, u in zip(inter_a, union_a)])
per_image_b = np.array([miou_from_counts(i, u) for i, u in zip(inter_b, union_b)])

print(f"Model A (CE + Dice): val mIoU = {miou_a:.4f}")
print(f"Model B (CE only):   val mIoU = {miou_b:.4f}")

# Per-class IoU table for the report
per_class = pd.DataFrame({
    'class': CLASSES,
    'IoU_A': inter_a.sum(0)[1:] / np.maximum(union_a.sum(0)[1:], 1),
    'IoU_B': inter_b.sum(0)[1:] / np.maximum(union_b.sum(0)[1:], 1),
})
display(per_class.round(3))

# The better model becomes the submitted best_segmentation.pth
best_name = "A" if miou_a >= miou_b else "B"
best_seg_path = seg_path_a if best_name == "A" else seg_path_b
shutil.copy2(best_seg_path, SEG_WEIGHTS / "best_segmentation.pth")
print(f"Selected model {best_name} -> {SEG_WEIGHTS / 'best_segmentation.pth'}")

# Save everything the statistical tests need (same validation images for A and B)
np.savez(OUT_DIR / "seg_val_results.npz", image_ids=np.array(VAL_IDS),
         inter_a=inter_a, union_a=union_a, inter_b=inter_b, union_b=union_b)
print("Saved", OUT_DIR / "seg_val_results.npz")

# Statistical tests

Both tests use the held-out validation images only (none of them were used for training either model).

**Wilcoxon signed-rank test (A vs B).** $d_i = \mathrm{mIoU}_{A,i} - \mathrm{mIoU}_{B,i}$ for each validation image $i$.
- $H_0$: the median of $d_i$ is 0 (no difference between A and B)
- $H_1$: the median of $d_i$ is not 0 (two-sided), $\alpha = 0.05$

Wilcoxon is used instead of a paired t-test because per-image IoU is bounded in [0, 1], and the differences are skewed with occasional large failures. Wilcoxon uses only the signs and ranks of the differences, so it doesn't need them to be normal and is less affected by outliers.

**Bootstrap CI.** For the best model, resample the validation images with replacement (B = 1000), recompute the dataset-level mIoU each time, and take the 2.5th and 97.5th percentiles.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# -------------------------------------------------------------
# 1. Wilcoxon signed-rank test on per-image mIoU
# -------------------------------------------------------------
ok = ~np.isnan(per_image_a) & ~np.isnan(per_image_b)
diffs = per_image_a[ok] - per_image_b[ok]
w_result = wilcoxon(per_image_a[ok], per_image_b[ok], alternative='two-sided')
print(f"Images compared: {ok.sum()}")
print(f"Mean per-image mIoU  A: {per_image_a[ok].mean():.4f} | B: {per_image_b[ok].mean():.4f}")
print(f"Wilcoxon statistic: {w_result.statistic:.1f} | p-value: {w_result.pvalue:.4g}")
print("Reject H0 (alpha=0.05)" if w_result.pvalue < 0.05 else "Fail to reject H0 (alpha=0.05)")

plt.figure(figsize=(7, 4))
plt.hist(diffs, bins=30, edgecolor='black')
plt.axvline(0, color='black', linestyle=':')
plt.axvline(np.mean(diffs), color='crimson', linestyle='--', label=f'mean = {np.mean(diffs):.4f}')
plt.title('Paired differences: mIoU_A - mIoU_B (per validation image)')
plt.xlabel('Difference'); plt.ylabel('Count'); plt.legend(); plt.tight_layout(); plt.show()

# -------------------------------------------------------------
# 2. Bootstrap 95% CI for the best model's dataset-level mIoU
# -------------------------------------------------------------
inter_best, union_best = (inter_a, union_a) if best_name == "A" else (inter_b, union_b)
rng = np.random.default_rng(SEED)
B = 1000
n = len(inter_best)
boot = np.array([
    miou_from_counts(inter_best[idx].sum(0), union_best[idx].sum(0))
    for idx in (rng.integers(0, n, size=n) for _ in range(B))
])
overall = miou_from_counts(inter_best.sum(0), union_best.sum(0))
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
print(f"\nBest model: {best_name}")
print(f"Overall val mIoU:     {overall:.4f}")
print(f"Bootstrap mean:       {boot.mean():.4f}")
print(f"Bootstrap std. error: {boot.std(ddof=1):.4f}")
print(f"95% CI (percentile):  [{ci_low:.4f}, {ci_high:.4f}]")

pd.DataFrame([{
    'wilcoxon_statistic': w_result.statistic, 'wilcoxon_pvalue': w_result.pvalue,
    'miou_A': miou_a, 'miou_B': miou_b, 'best_model': best_name,
    'bootstrap_B': B, 'bootstrap_mean': boot.mean(), 'bootstrap_se': boot.std(ddof=1),
    'ci_low': ci_low, 'ci_high': ci_high, 'n_val_images': n,
}]).to_csv(OUT_DIR / "statistical_test_results.csv", index=False)
print("Saved", OUT_DIR / "statistical_test_results.csv")

# Submission

`submission.csv` is made by loading both `model.py` files from disk, exactly as the grader will, and predicting every test image.

In [ ]:
# Generate submission.csv with the submitted model.py files
cls_model = load_module("cls_final", CLS_DIR / "model.py").ClassificationModel(str(CLS_WEIGHTS))
seg_model = load_module("seg_final", SEG_DIR / "model.py").SegmentationModel(str(SEG_WEIGHTS))

test_paths = sorted(TEST_IMG_DIR.glob("*.jpg"))
if SMOKE_TEST:
    test_paths = test_paths[:5]

rows = []
for k, path in enumerate(test_paths, start=1):
    with Image.open(path) as img:
        image = np.array(img.convert("RGB"))
    probs = cls_model.predict(image)
    labels = [c for c in CLASSES if probs[c] >= 0.5]
    mask = seg_model.predict(image)
    assert mask.shape == image.shape[:2] and mask.dtype == np.uint8
    rle = mask_to_rle(mask)
    assert np.array_equal(rle_to_mask(rle, *mask.shape), mask)   # RLE round-trips
    rows.append({'image_id': path.stem,
                 'classification': ' '.join(labels) if labels else 'background',
                 'segmentation_rle': rle})
    if k % 100 == 0 or k == len(test_paths):
        print(f"Processed {k}/{len(test_paths)}")

submission = pd.DataFrame(rows, columns=['image_id', 'classification', 'segmentation_rle'])
submission.to_csv(OUT_DIR / "submission.csv", index=False)
print("Saved", OUT_DIR / "submission.csv", "|", len(submission), "rows")
display(submission.head())

In [ ]:
# Copy requirements.txt into the submission folder and list what is there
(OUT_DIR / "requirements.txt").write_text("numpy\nPillow\ntorch\ntorchvision\npandas\nscipy\nmatplotlib\n")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUT_DIR)}  ({p.stat().st_size / 1e6:.1f} MB)")